# 01: Cleaning Bronze to Silver

**Exam objective (from Data Transformation and Modeling domain):** 
Implement data cleaning by reading bronze tables with PySpark/SQL, 
cleaning nulls, standardizing data types, and writing to new silver 
tables.

**Scope:** DataFrame-level cleaning operations. This exercise covers 
the standard bronze-to-silver transition: null handling, type casting, 
column renaming, and the write patterns for silver tables.

In [0]:
# Create a bronze table with sample data to practice data cleansing scenarios
from pyspark.sql import Row

bronze_data = [
    Row(order_id="1001", customer_id="C001",  order_date="2026-06-15", amount="149.99",  status="COMPLETED",  region="north"),
    Row(order_id="1002", customer_id="C002",  order_date="2026-06-15", amount="89.50",   status="completed",  region="South"),
    Row(order_id="1003", customer_id=None,    order_date="2026-06-16", amount="245.00",  status="PENDING",    region="east"),
    Row(order_id="1004", customer_id="C004",  order_date="2026/06/16", amount="52.25",   status="Completed",  region="west"),
    Row(order_id="1005", customer_id="C005",  order_date="2026-06-17", amount=None,      status="COMPLETED",  region=None),
    Row(order_id="1006", customer_id="C006",  order_date="2026-06-17", amount="1875.00", status="completed",  region="North"),
    Row(order_id="1007", customer_id="C007",  order_date=None,         amount="199.99",  status="CANCELLED",  region="south"),
    Row(order_id="1008", customer_id="C008",  order_date="2026-06-18", amount="75.00",   status="pending",    region="East"),
    Row(order_id="1001", customer_id="C001",  order_date="2026-06-15", amount="149.99",  status="COMPLETED",  region="north"),  # exact duplicate of first row
    Row(order_id="1009", customer_id="C009",  order_date="2026-06-18", amount="329.50",  status="Completed",  region="West"),
    Row(order_id=None,   customer_id="C010",  order_date="2026-06-19", amount="88.00",   status="COMPLETED",  region="north"),
    Row(order_id="1011", customer_id="C011",  order_date="2026-06-19", amount="425.75",  status="Pending",    region="south"),
]

bronze_df = spark.createDataFrame(bronze_data)

# Write as the bronze table
(bronze_df.write
    .mode("overwrite")
    .saveAsTable("certprep.transformations.orders_bronze")
)

display(spark.table("certprep.transformations.orders_bronze"))

## The bronze table

The bronze `orders_bronze` table represents raw ingested data from a hypothetical order-processing system. Quality issues present:

- All columns are strings, including numeric (amount) and date (order_date).
- Inconsistent case in status: COMPLETED/completed/Completed.
- Inconsistent case in region: north/North/South/etc..
- Inconsistent date format: one row uses slashes, the rest use hyphens.
- Nulls in customer_id, amount, region, order_date, and order_id.
- An exact duplicate row.

Each of these requires a cleaning decision in the bronze-to-silver transition. The next steps will handle them one category at a time.

## Null handling operations

- `df.dropna()` — drop rows with any nulls in any column
- `df.dropna(subset=['col1', 'col2'])` — drop rows with nulls in specific columns
- `df.dropna(how='any')` (default) — drop if any listed column is null
- `df.dropna(how='all')` — drop only if all listed columns are null
- `df.fillna(value)` — fill all nulls with a single value (type-appropriate)
- `df.fillna({'col1': 'unknown', 'col2': 0})` — fill different columns with different defaults
- `df.na.drop(...)` and `df.na.fill(...)` — alternative syntax for the same operations

In [0]:
# null handling
from pyspark.sql import functions as F

# read bronze table
bronze_df = spark.table("certprep.transformations.orders_bronze")

print(f"Rows before null handling: {bronze_df.count()}")

# drop rows with nulls in critical columns
# critical columns in this scenario: order_id, customer_id, order_date, & amount
cleaned_df = bronze_df.dropna(subset=["order_id", "customer_id", "order_date", "amount"])

print(f"Rows after null handling: {cleaned_df.count()}")

cleaned_df = cleaned_df.fillna({'region': 'unknown'})

display(cleaned_df)

## Why drop vs. fill?

For each null column in this exercise, the rule I applied was:

- **order_id, customer_id, order_date, amount**: dropped. These are identifiers or primary metrics — a row missing any of them can't be meaningfully used downstream. Filling with a default would fabricate data.

- **region**: filled with 'unknown'. Region is categorical and used for grouping in analytics. Filling with 'unknown' lets those rows still contribute to overall totals while being explicitly distinguishable from known regions in group-by analysis.

The general principle: **drop when the null makes the row unusable, fill when a sensible default preserves the row's usefulness, leave null when downstream logic handles nulls correctly.**

In [0]:
# datatype standardization for silver layer
from pyspark.sql.types import IntegerType, DecimalType

cleaned_df = (cleaned_df
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("amount", F.col("amount").cast(DecimalType(10, 2)))
    .withColumn("order_date", F.to_date(F.regexp_replace(F.col("order_date"), "/", "-"), "yyyy-MM-dd"))
)

# verify updated datatypes 
cleaned_df.printSchema()
display(cleaned_df)

## Type standardization operations

- `col.cast(type)` — simple type conversion. Works when the source string is parseable as the target type without transformation. Examples: "123" → INT, "45.67" → DECIMAL.
- `to_date(col, format)` — parse a string to a DATE using a format specifier. Returns null if the string doesn't match the format.
- `to_timestamp(col, format)` — same for TIMESTAMP.
- `coalesce(a, b, c)` — return the first non-null value. Useful for trying multiple date formats in sequence.
- `regexp_replace(col, pattern, replacement)` — string transformation before casting, when the input has a fixable format inconsistency.

### On the date format issue

The bronze data has dates in mostly ISO format (`2026-06-15`) with one row using slashes (`2026/06/16`). Two ways to handle this:

1. Coalesce multiple `to_date` calls with different format specifiers.
2. Normalize the string with `regexp_replace` first, then cast once.

Option 2 is cleaner when the fix is simple. Option 1 is more explicit about which formats are considered valid.

In [0]:
# standardize string column values via case 
cleaned_df = (cleaned_df
              .withColumn("status", F.upper(F.trim(F.col("status"))))
              .withColumn("region", F.lower(F.trim(F.col("region"))))
)

display(cleaned_df)

## Categorical standardization operations

Case normalization:
- `upper(col)` — convert to uppercase
- `lower(col)` — convert to lowercase
- `initcap(col)` — capitalize first letter of each word

Whitespace normalization:
- `trim(col)` — remove leading and trailing whitespace
- `ltrim(col)` — remove leading only
- `rtrim(col)` — remove trailing only

Value mapping (when raw values need to be replaced with canonical ones):
- `when(condition, value).when(...).otherwise(default)` — SQL-style CASE
- `regexp_replace(col, pattern, replacement)` — regex-based substitution

### The naming/case convention

For this exercise:
- Enum-like categorical columns (status codes, flags) → uppercase.
- Descriptive categorical columns (region names, product names) → lowercase.

This is convention, not required. Teams pick a house style and enforce it consistently — the important thing is that raw values are normalized before landing in silver, so downstream queries can rely on canonical forms.

In [0]:
# drop exact duplicates
print(f"Rows before deduplication: {cleaned_df.count()}")

cleaned_df = cleaned_df.dropDuplicates()

print(f"Rows after deduplication: {cleaned_df.count()}")
display(cleaned_df)

## Deduplication operations

- `dropDuplicates()` — removes rows identical across all columns.
- `dropDuplicates(subset=[cols])` — removes rows duplicated on the 
  specified columns, keeping the first occurrence arbitrarily.
- Window function pattern (`row_number()` + `filter`) — when you need 
  explicit control over which duplicate to keep based on a tiebreaker.

Window function example:
```python
from pyspark.sql import Window

w = Window.partitionBy("order_id").orderBy(F.col("ingested_at").desc())

deduped = (df
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
```

### Choosing the right dedup pattern

- **Exact duplicates** (all columns identical): `dropDuplicates()` alone.
- **Business-key duplicates with arbitrary winner**: 
  `dropDuplicates(subset=[keys])`.
- **Business-key duplicates with specific winner** (latest, highest, etc.): 
  window function with `row_number()` and `filter(row_num == 1)`.

The "arbitrary winner" of `dropDuplicates(subset=...)` is a hazard in 
production — you don't control which row wins, and non-deterministic 
Spark operations can produce different results across runs. If which 
duplicate wins matters for correctness, use the window function pattern.

In [0]:
# rename columns for silver table write
cleaned_df = cleaned_df.withColumnsRenamed({
    "amount" : "order_amount",
    "status" : "order_status",
    "region" : "sales_region"
})

cleaned_df.printSchema()
display(cleaned_df)

In [0]:
# write to silver
(cleaned_df.write
    .mode("overwrite")
    .saveAsTable("certprep.transformations.orders_silver")
)

display(spark.table("certprep.transformations.orders_silver"))

In [0]:
# verify write
silver = spark.table("certprep.transformations.orders_silver")

# verify schema
print("Silver schema:")
silver.printSchema()

# verify no unexpected nulls
print("\nNull counts per column:")
silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver.columns
]).show()

# verify categorical standardization
print("\nDistinct order_status values:")
silver.select("order_status").distinct().show()

print("\nDistinct sales_region values:")
silver.select("sales_region").distinct().show()

## Column renaming operations

- `withColumnRenamed(old, new)` — rename one column.
- `withColumnsRenamed({old1: new1, old2: new2})` — rename multiple columns in one call.

## Write modes

- `overwrite`: replaces the entire target table. Used for full loads.
- `append`: adds new rows to the existing target table. Used for incremental loads.
- `ignore`: does nothing if the target exists. Rarely used.
- `error` (default): fails if the target exists.

For a bronze-to-silver clean of a full dataset, `overwrite` is standard. For incremental cleaning where new bronze rows are processed and added to silver, `append` is standard.

1. You have a bronze table with the following columns and issues:
   - `customer_id`: primary key, occasionally null
   - `email`: often null but not always required downstream
   - `signup_date`: string in inconsistent formats
   - `total_spend`: decimal-looking string with occasional nulls
   
   Describe the cleaning strategy for each column. For nulls, state whether you'd drop, fill, or leave, and why. For type issues, state what operation you'd use.

2. A colleague writes this cleaning code:
```python
   cleaned = bronze.dropna().fillna({'region': 'unknown'})
```
   What's wrong with this, and what would the correct version look like?

3. You need to deduplicate an orders table on `order_id`. Two rows exist with `order_id = 5001` — one has `status = 'PENDING'` and `ingested_at = '2026-06-15 10:00'`, the other has `status = 'COMPLETED'` and `ingested_at = '2026-06-15 14:00'`. You want to keep the row representing the latest state. Which dedup approach do you use, and why not `dropDuplicates(subset=['order_id'])`?

4. Your cleaning pipeline casts a string column to INT using `col.cast(IntegerType())`. A downstream analyst reports that some values that should be numeric are showing as null in the silver table. What likely happened, and how would you catch this earlier in the pipeline?

5. You're standardizing a `country_code` column. The bronze data contains values like `'us'`, `'US'`, `' US '`, `'usa'`, `'United States'`. Simple `upper(trim(col))` would still leave you with `'US'`, `'USA'`, `'UNITED STATES'` — three distinct values for the same concept. What pattern would you use to normalize all of these to a single canonical value?

6. Write modes: your silver-layer job runs nightly. Each night, it processes yesterday's newly-arrived bronze rows and produces silver rows for those specific rows. Which write mode do you use for the silver write? What write mode would you use if instead the job reprocessed the entire bronze table every night?

## Self Check Answers

1. For the `customer_id` column, I would drop the nulls with `df.dropna(subset=["customer_id"])` as the column is a primary key and cannot allow nulls, nor can I fill in a value given uniquieness and data fabrication constraints. Given that the `email` column is not always required downstream, I would use `df.fillna({'email': 'Unknown'})` to set "Unknown" as a default value. `signup_date` will require two things: using `regexp_replace()` to standardize different date patterns to a single pattern (will require examining the data), followed by casting the column as a date or datetime datatype. Lastly, `total_spend` should be cast as a `DECIMALTYPE(10,2)`, but depending on what exactly this data is tracking, I'd suggest either dropping the rows with a null value (ex: tracking how much was spent in a transaction) or fill the null with a default value of 0.0. In this case, I'd lean more towards dropping the row if I assume the column holds data relevant to downstream analytics. 
2. This issue with this code is that running `dropna()` before `fillna()` renders the latter function useless as well as results in data loss whereever the `region` column has null values. The correct version looks like this:
    ```python
    cleaned = bronze.dropna(subset=[/* droppable columns here */]).fillna({'region': 'unknown'})
    ```
3. I would use a window function to get the latest state via `row_number()` and filter out the rest. Using `dropDuplicates(subset=['order_id'])` is an unsafe option because determining which row to keep vs drop is arbitrary and risks ending up with inacurate data. 
4. If casting is resulting in nulls, it means the initial value cannot cast to the specified datatype. Since the casted column is a string column, `trim()` can be used to eliminate any leading or trailing spaces for starters. It could be an issue of size, depending on the column and its values, where `IntegerType()` is too small a range in some cases. Such would warrant updating the cast to `LongType()` for larger numbers. Ultimately depends on the data, further cleaning may also be warranted if unexpected characters such as letters or symbols occur in the initial string. Another strategy is counting nulls before and after the cast to catch any edge cases. 
5. Using `upper` and `trim` is correct, but given the variants using `regexp_replace` to consolidate the expected variants to a single expression to the missing step here.
6. Use `.mode("append")` in this case for incremental loading. If reprocessing the entire bronze table into silver, use `.mode("overwrite")`.